# Model Training and Evaluation

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

from src.models.lgbm_model import LGBMModel
from src.models.xgb_model import XGBModel
from src.models.catboost_model import CatBoostModel
from src.models.ensemble import weighted_average_ensemble, optimize_weights
from src.evaluation.metrics import rmsle
from src.evaluation.validation import TimeSeriesSplitWithGap
from src.config import (
    LGBM_PARAMS, XGB_PARAMS, CATBOOST_PARAMS,
    N_SPLITS, VALIDATION_GAP_DAYS, HOLDOUT_DAYS, SEED
)

print('Imports successful.')

In [ ]:
# Demonstrate data loading pattern (without training on full data)
from pathlib import Path

DATA_DIR = Path('../data/raw')
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])

# Show sample of data structure that will be used for training
print('Training data shape:', train.shape)
print('Date range:', train['date'].min(), 'to', train['date'].max())
print('Unique stores:', train['store_nbr'].nunique())
print('Unique families:', train['family'].nunique())
print('\nSample data:')
display(train.head())

In [ ]:
# Demonstrate TimeSeriesSplitWithGap
splitter = TimeSeriesSplitWithGap(
    n_splits=N_SPLITS,
    gap_days=VALIDATION_GAP_DAYS
)

# Use a small synthetic date index to show split structure
dates = pd.date_range('2016-01-01', '2017-08-15', freq='D')
X_demo = pd.DataFrame({'date': dates, 'value': np.random.rand(len(dates))})

print(f'Total samples: {len(X_demo)}')
print(f'N splits: {N_SPLITS}')
print(f'Gap days: {VALIDATION_GAP_DAYS}')
print()

for fold_idx, (train_idx, val_idx) in enumerate(splitter.split(X_demo)):
    train_dates = X_demo.iloc[train_idx]['date']
    val_dates = X_demo.iloc[val_idx]['date']
    print(f'Fold {fold_idx+1}:')
    print(f'  Train: {train_dates.min().date()} to {train_dates.max().date()} ({len(train_idx)} days)')
    print(f'  Val:   {val_dates.min().date()} to {val_dates.max().date()} ({len(val_idx)} days)')

In [ ]:
print('=== LGBM_PARAMS ===')
for k, v in LGBM_PARAMS.items():
    print(f'  {k}: {v}')

print('\n=== XGB_PARAMS ===')
for k, v in XGB_PARAMS.items():
    print(f'  {k}: {v}')

print('\n=== CATBOOST_PARAMS ===')
for k, v in CATBOOST_PARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
# Demonstrate ensemble strategy
print('=== Ensemble Strategy ===')
print()
print('weighted_average_ensemble(predictions, weights):')
print('  - Combines model predictions by weighted sum')
print('  - Clips output to >= 0 (no negative sales)')
print()
print('optimize_weights(predictions, y_true):')
print('  - Uses scipy.optimize.minimize to find optimal weights')
print('  - Minimizes RMSLE on hold-out validation set')
print('  - Constraint: weights sum to 1, each weight >= 0')
print()

# Synthetic demonstration
np.random.seed(SEED)
y_true_demo = np.random.rand(100) * 50
pred_lgbm = y_true_demo + np.random.randn(100) * 3
pred_xgb = y_true_demo + np.random.randn(100) * 4
pred_cat = y_true_demo + np.random.randn(100) * 5

# Show individual RMSLE scores
print('Individual model RMSLE (synthetic demo):')
print(f'  LGBM:     {rmsle(y_true_demo, pred_lgbm.clip(0)):.4f}')
print(f'  XGBoost:  {rmsle(y_true_demo, pred_xgb.clip(0)):.4f}')
print(f'  CatBoost: {rmsle(y_true_demo, pred_cat.clip(0)):.4f}')

# Simple equal-weight ensemble
equal_ensemble = weighted_average_ensemble([pred_lgbm, pred_xgb, pred_cat], [1/3, 1/3, 1/3])
print(f'  Equal-weight ensemble: {rmsle(y_true_demo, equal_ensemble):.4f}')

## Validation Strategy
TimeSeriesSplit with 16-day gap prevents leakage